# Global-normalized InstanSeg WSI → watershed Zarr

This experimental notebook runs the fork's global-normalized whole-slide method with native reconciliation disabled, then reconciles the unresolved model-resolution nuclear and cell planes directly from Zarr. It never constructs a whole-slide multichannel float array. Final native-resolution TIFF export is optional and occurs only after structural validation.


In [1]:
from pathlib import Path
import gc, json, os, re, subprocess, sys, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.segmentation import watershed
import tifffile
import torch
import zarr

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
INSTANSEG_ROOT = (REPO_ROOT.parent / 'instanseg').resolve()
if not (INSTANSEG_ROOT / 'instanseg/inference_class.py').exists():
    raise FileNotFoundError(INSTANSEG_ROOT)
sys.path.insert(0, str(INSTANSEG_ROOT))
from instanseg import InstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as instanseg_inference_class
instanseg_inference_class.TiffSlide = TiffSlide

INSTANSEG_HEAD = subprocess.run(
    ['git', '-C', str(INSTANSEG_ROOT), 'rev-parse', 'HEAD'],
    check=True, text=True, capture_output=True,
).stdout.strip()
assert hasattr(InstanSeg, 'eval_whole_slide_image_global_normalization')
print({'python': sys.executable, 'instanseg_root': str(INSTANSEG_ROOT), 'instanseg_head': INSTANSEG_HEAD, 'cuda': torch.cuda.is_available()})


{'python': '/data1/lowes/ratnayn/conda_envs/instanseg_nimbus/bin/python', 'instanseg_root': '/data1/lowes/ratnayn/Codex/projects/instanseg', 'instanseg_head': 'a705b26fadf045225d47ca006b9d2269339b4fc8', 'cuda': True}


In [2]:
# Inputs and explicit execution gates.
SLIDE_ID = 'SLIDE-0330'
FULL_MERGE_OME = Path(
    '/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/'
    'SLIDE-0330/outputs_v3/SLIDE-0330_full_merge.ome.tif'
)
SCRATCH_ROOT = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline')
OUTPUT_DIR = SCRATCH_ROOT / 'instanseg_watershed_wsi_global_testbatch' / SLIDE_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_UNRESOLVED_ZARR = OUTPUT_DIR / f'{SLIDE_ID}_unresolved_global_wsi.zarr'
RECONCILED_ZARR = OUTPUT_DIR / f'{SLIDE_ID}_watershed_reconciled.zarr'
WORK_CELL_DAT = OUTPUT_DIR / f'{SLIDE_ID}_work_cells.uint32.dat'
WORK_NUCLEAR_DAT = OUTPUT_DIR / f'{SLIDE_ID}_work_nuclei.uint32.dat'
FINAL_CELL_TIFF = OUTPUT_DIR / f'{SLIDE_ID}_watershed_whole_cell.tiff'
FINAL_NUCLEAR_TIFF = OUTPUT_DIR / f'{SLIDE_ID}_watershed_nuclear.tiff'
PROVENANCE_JSON = OUTPUT_DIR / f'{SLIDE_ID}_experiment.json'

RUN_WSI_INFERENCE = True
RUN_WATERSHED_RECONCILIATION = True
WRITE_RECONCILED_ZARR = True
WRITE_NATIVE_TIFFS = True
OVERWRITE_WSI = True
REBUILD_WORK_ARRAYS = True
ROW_CHUNK = 512

SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_F480_D2S9R_555', 'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
REFERENCE_CHANNEL = 'R1_DAPI'
MODEL_NAME = 'fluorescence_nuclei_and_cells'
PIXEL_SIZE_UM = 0.325
WSI_TILE_SIZE = 2048
WSI_OVERLAP = 80
WSI_DETECTION_SIZE = 20
WSI_INFERENCE_BATCH_SIZE = 4
COMPARE_BATCH_MODES = True
REFERENCE_BATCH_SIZE = 1
COMPARISON_BATCH_SIZE = WSI_INFERENCE_BATCH_SIZE
if COMPARE_BATCH_MODES and COMPARISON_BATCH_SIZE == REFERENCE_BATCH_SIZE:
    raise ValueError('Choose a comparison batch size different from 1.')
REFERENCE_UNRESOLVED_ZARR = OUTPUT_DIR / f'{SLIDE_ID}_unresolved_global_wsi_batch{REFERENCE_BATCH_SIZE}.zarr'
BATCHED_UNRESOLVED_ZARR = OUTPUT_DIR / f'{SLIDE_ID}_unresolved_global_wsi_batch{COMPARISON_BATCH_SIZE}.zarr'
COMPARISON_REPORT_JSON = OUTPUT_DIR / f'{SLIDE_ID}_unresolved_global_wsi_batch_comparison.json'
UNRESOLVED_ZARR = BATCHED_UNRESOLVED_ZARR if COMPARE_BATCH_MODES else DEFAULT_UNRESOLVED_ZARR
NORMALIZATION_PERCENTILES = (0.1, 99.9)
EVAL_KWARGS = {
    'min_size': 10, 'mask_threshold': 0.53, 'peak_distance': 5,
    'seed_threshold': 0.7, 'overlap_threshold': 0.3,
    'mean_threshold': 0.0, 'fg_threshold': 0.5, 'window_size': 32,
    'cleanup_fragments': True, 'resolve_cell_and_nucleus': False,
}


In [3]:
def channel_names_from_ome_xml(xml):
    return re.findall(r'<Channel[^>]*Name="([^"]+)"', xml or '')

with tifffile.TiffFile(FULL_MERGE_OME) as handle:
    source_names = channel_names_from_ome_xml(handle.ome_metadata)
    source_shape = tuple(int(v) for v in handle.series[0].shape)
    native_shape = tuple(int(v) for v in handle.series[0].levels[0].pages[0].shape[-2:])
missing = [name for name in SEGMENTATION_CHANNELS if name not in source_names]
if missing:
    raise ValueError(f'Missing segmentation channels: {missing}')
channel_to_index = {name: index for index, name in enumerate(source_names)}
CHANNEL_IDS = [channel_to_index[name] for name in SEGMENTATION_CHANNELS]
REFERENCE_CHANNEL_ID = channel_to_index[REFERENCE_CHANNEL]
assert REFERENCE_CHANNEL_ID in CHANNEL_IDS
PROVENANCE = {
    'slide_id': SLIDE_ID, 'source_image': str(FULL_MERGE_OME),
    'source_shape': list(source_shape), 'native_shape': list(native_shape),
    'channels': SEGMENTATION_CHANNELS, 'channel_ids': CHANNEL_IDS,
    'reference_channel': REFERENCE_CHANNEL, 'reference_channel_id': REFERENCE_CHANNEL_ID,
    'model': MODEL_NAME, 'pixel_size_um': PIXEL_SIZE_UM,
    'normalization_percentiles': list(NORMALIZATION_PERCENTILES),
    'tile_size': WSI_TILE_SIZE, 'overlap': WSI_OVERLAP,
    'detection_size': WSI_DETECTION_SIZE, 'batch_size': WSI_INFERENCE_BATCH_SIZE,
    'compare_batch_modes': COMPARE_BATCH_MODES,
    'comparison_batch_sizes': [REFERENCE_BATCH_SIZE, COMPARISON_BATCH_SIZE],
    'comparison_zarrs': [str(REFERENCE_UNRESOLVED_ZARR), str(BATCHED_UNRESOLVED_ZARR)],
    'eval_kwargs': EVAL_KWARGS,
    'instanseg_root': str(INSTANSEG_ROOT), 'instanseg_head': INSTANSEG_HEAD,
    'unresolved_zarr': str(UNRESOLVED_ZARR),
}
PROVENANCE_JSON.write_text(json.dumps(PROVENANCE, indent=2) + '\n')
display(PROVENANCE)


{'slide_id': 'SLIDE-0330',
 'source_image': '/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/SLIDE-0330/outputs_v3/SLIDE-0330_full_merge.ome.tif',
 'source_shape': [72, 55388, 62688],
 'native_shape': [55388, 62688],
 'channels': ['R1_DAPI',
  'R4_P19_POLYRAT',
  'R4_GFP_POLY_AF488',
  'R6_CD45_CST_AF647',
  'R6_PANCK_AE1_AE3_750',
  'R12_CD31_D8V9E_AF750',
  'R7_NAK_ATPASE_555',
  'R8_F480_D2S9R_555',
  'R9_CD68_E3O7V_488',
  'R12_CD3E_E4T1B_AF555'],
 'channel_ids': [4, 38, 42, 51, 52, 17, 56, 62, 71, 15],
 'reference_channel': 'R1_DAPI',
 'reference_channel_id': 4,
 'model': 'fluorescence_nuclei_and_cells',
 'pixel_size_um': 0.325,
 'normalization_percentiles': [0.1, 99.9],
 'tile_size': 2048,
 'overlap': 80,
 'detection_size': 20,
 'batch_size': 4,
 'compare_batch_modes': True,
 'comparison_batch_sizes': [1, 4],
 'comparison_zarrs': ['/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_wsi_global_testbatch/SLIDE-0330/SLIDE-0330_unresolved_global_wsi

## Run or reuse globally normalized WSI inference

The Zarr is accepted for reuse only when its completion flag, source image, channel order, pixel size, and unresolved setting match this experiment.


In [ ]:
def run_wsi_inference(batch_size, output_path):
    inst = InstanSeg(MODEL_NAME, verbosity=1)
    started = time.perf_counter()
    observed_path = inst.eval_whole_slide_image_global_normalization(
        str(FULL_MERGE_OME), channel_ids=CHANNEL_IDS, pixel_size=PIXEL_SIZE_UM,
        normalization_percentiles=NORMALIZATION_PERCENTILES,
        reference_channel_id=REFERENCE_CHANNEL_ID, tile_size=WSI_TILE_SIZE,
        overlap=WSI_OVERLAP, detection_size=WSI_DETECTION_SIZE,
        batch_size=batch_size, output_path=output_path,
        overwrite=OVERWRITE_WSI, **EVAL_KWARGS,
    )
    elapsed = time.perf_counter() - started
    print({'batch_size': batch_size, 'minutes': elapsed / 60, 'path': str(observed_path)})
    return observed_path, elapsed

def compare_wsi_outputs(reference_path, candidate_path, row_chunk):
    reference = zarr.open(str(reference_path), mode='r')
    candidate = zarr.open(str(candidate_path), mode='r')
    if tuple(reference.shape) != tuple(candidate.shape):
        raise ValueError(f'WSI output shapes differ: {reference.shape} vs {candidate.shape}')
    if reference.dtype != candidate.dtype:
        raise ValueError(f'WSI output dtypes differ: {reference.dtype} vs {candidate.dtype}')
    plane_names = ['nuclei', 'cells'][:reference.shape[0]]
    total_pixels = reference.shape[-2] * reference.shape[-1]
    reports = []
    for plane, plane_name in enumerate(plane_names):
        exact_pixels = 0
        differing_pixels = 0
        reference_foreground = 0
        candidate_foreground = 0
        foreground_intersection = 0
        foreground_union = 0
        reference_max = 0
        candidate_max = 0
        for y0 in range(0, reference.shape[-2], row_chunk):
            y1 = min(reference.shape[-2], y0 + row_chunk)
            reference_chunk = np.asarray(reference[plane, y0:y1])
            candidate_chunk = np.asarray(candidate[plane, y0:y1])
            equal = reference_chunk == candidate_chunk
            reference_mask = reference_chunk > 0
            candidate_mask = candidate_chunk > 0
            exact_pixels += int(np.count_nonzero(equal))
            differing_pixels += int(np.count_nonzero(~equal))
            reference_foreground += int(np.count_nonzero(reference_mask))
            candidate_foreground += int(np.count_nonzero(candidate_mask))
            foreground_intersection += int(np.count_nonzero(reference_mask & candidate_mask))
            foreground_union += int(np.count_nonzero(reference_mask | candidate_mask))
            reference_max = max(reference_max, int(reference_chunk.max()))
            candidate_max = max(candidate_max, int(candidate_chunk.max()))
        reports.append({
            'plane': plane_name, 'exact_label_fraction': exact_pixels / total_pixels,
            'differing_label_pixels': differing_pixels,
            'reference_foreground_pixels': reference_foreground,
            'candidate_foreground_pixels': candidate_foreground,
            'foreground_iou': foreground_intersection / foreground_union if foreground_union else 1.0,
            'reference_max_label': reference_max, 'candidate_max_label': candidate_max,
        })
    return {'reference': str(reference_path), 'candidate': str(candidate_path), 'planes': reports}

if COMPARE_BATCH_MODES:
    if RUN_WSI_INFERENCE:
        candidate_path, candidate_seconds = run_wsi_inference(COMPARISON_BATCH_SIZE, BATCHED_UNRESOLVED_ZARR)
        reference_path, reference_seconds = run_wsi_inference(REFERENCE_BATCH_SIZE, REFERENCE_UNRESOLVED_ZARR)
    else:
        reference_path, candidate_path = REFERENCE_UNRESOLVED_ZARR, BATCHED_UNRESOLVED_ZARR
        reference_seconds = candidate_seconds = None
    if not reference_path.exists() or not candidate_path.exists():
        raise FileNotFoundError('Run both batch modes or provide both comparison Zarrs.')
    comparison_report = compare_wsi_outputs(reference_path, candidate_path, ROW_CHUNK)
    comparison_report.update({
        'reference_batch_size': REFERENCE_BATCH_SIZE,
        'candidate_batch_size': COMPARISON_BATCH_SIZE,
        'reference_seconds': reference_seconds, 'candidate_seconds': candidate_seconds,
    })
    COMPARISON_REPORT_JSON.write_text(json.dumps(comparison_report, indent=2) + '\n')
    display(comparison_report)
else:
    if RUN_WSI_INFERENCE:
        run_wsi_inference(WSI_INFERENCE_BATCH_SIZE, UNRESOLVED_ZARR)
if not UNRESOLVED_ZARR.exists():
    raise FileNotFoundError('Set RUN_WSI_INFERENCE=True or provide the completed unresolved Zarr.')
raw = zarr.open(str(UNRESOLVED_ZARR), mode='r')
assert raw.ndim == 3 and raw.shape[0] == 2 and raw.dtype == np.int32
assert raw.attrs['status'] == 'complete'
assert raw.attrs['source_image'] == str(FULL_MERGE_OME)
assert list(raw.attrs['channel_ids']) == CHANNEL_IDS
assert raw.attrs['planes'] == ['nuclei', 'cells']
assert raw.attrs['wsi_settings']['resolve_cell_and_nucleus'] is False
model_shape = tuple(int(v) for v in raw.shape[-2:])
display({'shape': raw.shape, 'chunks': raw.chunks, 'dtype': str(raw.dtype), 'attrs': dict(raw.attrs)})


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/utils/../bioimageio_models/, loading
Requesting default device: cuda


Slide progress:   1%|          | 3/440 [00:04<10:12,  1.40s/it]/data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/utils/pytorch_utils.py:312: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  intersection = torch.sparse.mm(onehot1, onehot2.T).to_dense()
Slide progress:  46%|████▌     | 202/440 [10:28<09:10,  2.31s/it]

## Copy unresolved planes into model-resolution working arrays

The raw Zarr stays immutable. Watershed edits disk-backed uint32 arrays copied one row chunk at a time.


In [ ]:
def copy_zarr_plane_to_memmap(source, plane, path, shape, row_chunk):
    target = np.memmap(path, mode='w+', dtype=np.uint32, shape=shape)
    for y0 in range(0, shape[0], row_chunk):
        y1 = min(shape[0], y0 + row_chunk)
        target[y0:y1] = np.asarray(source[plane, y0:y1], dtype=np.uint32)
    target.flush()
    return target

if RUN_WATERSHED_RECONCILIATION:
    if REBUILD_WORK_ARRAYS or not (WORK_CELL_DAT.exists() and WORK_NUCLEAR_DAT.exists()):
        final_nuclei = copy_zarr_plane_to_memmap(raw, 0, WORK_NUCLEAR_DAT, model_shape, ROW_CHUNK)
        final_cells = copy_zarr_plane_to_memmap(raw, 1, WORK_CELL_DAT, model_shape, ROW_CHUNK)
    else:
        final_nuclei = np.memmap(WORK_NUCLEAR_DAT, mode='r+', dtype=np.uint32, shape=model_shape)
        final_cells = np.memmap(WORK_CELL_DAT, mode='r+', dtype=np.uint32, shape=model_shape)
else:
    print('Set RUN_WATERSHED_RECONCILIATION=True to build or reopen working arrays.')


In [ ]:
def reduce_code_counts(code_parts, count_parts):
    codes = np.concatenate(code_parts); counts = np.concatenate(count_parts)
    order = np.argsort(codes); codes = codes[order]; counts = counts[order]
    starts = np.r_[0, np.flatnonzero(codes[1:] != codes[:-1]) + 1]
    return codes[starts], np.add.reduceat(counts, starts)

def compute_overlap_counts_chunked(cells, nuclei, row_chunk):
    max_cell = int(np.max(cells)); max_nucleus = int(np.max(nuclei))
    cell_areas = np.zeros(max_cell + 1, dtype=np.int64)
    nucleus_areas = np.zeros(max_nucleus + 1, dtype=np.int64)
    base = np.uint64(max_nucleus + 1); code_parts = []; count_parts = []
    for y0 in range(0, cells.shape[0], row_chunk):
        y1 = min(cells.shape[0], y0 + row_chunk)
        c = np.asarray(cells[y0:y1]); n = np.asarray(nuclei[y0:y1])
        cell_areas += np.bincount(c.ravel(), minlength=max_cell + 1)
        nucleus_areas += np.bincount(n.ravel(), minlength=max_nucleus + 1)
        both = (c > 0) & (n > 0)
        if np.any(both):
            codes, counts = np.unique(c[both].astype(np.uint64) * base + n[both].astype(np.uint64), return_counts=True)
            code_parts.append(codes); count_parts.append(counts.astype(np.int64))
    pair_codes, pair_counts = reduce_code_counts(code_parts, count_parts)
    return (cell_areas, nucleus_areas, (pair_codes // base).astype(np.uint32),
            (pair_codes % base).astype(np.uint32), pair_counts)

def combine_slices(slices, margin, shape):
    return (slice(max(0, min(s[0].start for s in slices) - margin), min(shape[0], max(s[0].stop for s in slices) + margin)),
            slice(max(0, min(s[1].start for s in slices) - margin), min(shape[1], max(s[1].stop for s in slices) + margin)))

def split_parent_local(cells, nuclei, cell_id, nucleus_ids, bbox):
    cell_crop = cells[bbox]; nucleus_crop = nuclei[bbox]
    parent = cell_crop == cell_id; territory = parent.copy()
    markers = np.zeros(parent.shape, dtype=np.int32)
    for marker_id, nucleus_id in enumerate(nucleus_ids, start=1):
        nucleus = nucleus_crop == nucleus_id; territory |= nucleus; markers[nucleus] = marker_id
    components, _ = ndi.label(territory)
    seeded_components = np.unique(components[markers > 0]); seeded_components = seeded_components[seeded_components > 0]
    seeded_territory = np.isin(components, seeded_components)
    split = watershed(-ndi.distance_transform_edt(seeded_territory), markers=markers, mask=seeded_territory)
    return parent, seeded_territory, markers, split


## Associate compartments and split only ambiguous cells


In [ ]:
if RUN_WATERSHED_RECONCILIATION:
    started = time.perf_counter()
    raw_cell_areas, raw_nucleus_areas, pair_cells, pair_nuclei, pair_pixels = compute_overlap_counts_chunked(final_cells, final_nuclei, ROW_CHUNK)
    qualifies = pair_pixels / raw_nucleus_areas[pair_nuclei] > 0.5
    qualifying_by_cell = {}
    for cell_id, nucleus_id in zip(pair_cells[qualifies].astype(int), pair_nuclei[qualifies].astype(int)):
        qualifying_by_cell.setdefault(cell_id, []).append(nucleus_id)
    raw_cell_ids = np.flatnonzero(raw_cell_areas[1:]) + 1
    raw_nucleus_ids = np.flatnonzero(raw_nucleus_areas[1:]) + 1
    matched = np.zeros(len(raw_nucleus_areas), dtype=bool); matched[pair_nuclei[qualifies]] = True
    unmatched_nucleus_ids = raw_nucleus_ids[~matched[raw_nucleus_ids]]
    ambiguous_cell_ids = np.array([c for c, nuclei in qualifying_by_cell.items() if len(nuclei) > 1], dtype=np.int64)
    association_summary = {
        'raw_cells': int(len(raw_cell_ids)), 'raw_nuclei': int(len(raw_nucleus_ids)),
        'single_nucleus_cells': int(sum(len(n) == 1 for n in qualifying_by_cell.values())),
        'ambiguous_cells': int(len(ambiguous_cell_ids)), 'unmatched_nuclei': int(len(unmatched_nucleus_ids)),
        'seconds': time.perf_counter() - started,
    }
    display(association_summary)


In [ ]:
if RUN_WATERSHED_RECONCILIATION:
    max_raw_cell_id = len(raw_cell_areas) - 1; max_raw_nucleus_id = len(raw_nucleus_areas) - 1
    cell_slices = ndi.find_objects(final_cells, max_label=max_raw_cell_id)
    nucleus_slices = ndi.find_objects(final_nuclei, max_label=max_raw_nucleus_id)
    nucleus_to_final_cell = np.zeros(max_raw_nucleus_id + 1, dtype=np.uint32)
    next_cell_id = max_raw_cell_id + 1; watershed_records = []
    for cell_id, associated in qualifying_by_cell.items():
        if len(associated) == 1:
            nucleus_to_final_cell[associated[0]] = cell_id
            continue
        objects = [cell_slices[cell_id - 1]] + [nucleus_slices[n - 1] for n in associated]
        if not all(item is not None for item in objects):
            raise RuntimeError(f'Missing bounding box for cell {cell_id}')
        bbox = combine_slices(objects, margin=8, shape=model_shape)
        parent, seeded_territory, markers, split = split_parent_local(final_cells, final_nuclei, cell_id, associated, bbox)
        daughter_ids = [cell_id] + list(range(next_cell_id, next_cell_id + len(associated) - 1))
        next_cell_id += len(associated) - 1
        local = final_cells[bbox]; local[parent & seeded_territory] = 0
        for marker_id, (nucleus_id, daughter_id) in enumerate(zip(associated, daughter_ids), start=1):
            local[(split == marker_id) & parent] = daughter_id
            nucleus_to_final_cell[nucleus_id] = daughter_id
        watershed_records.append({
            'parent_cell_id': int(cell_id), 'nucleus_ids': [int(v) for v in associated],
            'daughter_ids': [int(v) for v in daughter_ids], 'bbox': bbox,
            'unseeded_parent_pixels_preserved': int((parent & ~seeded_territory).sum()),
            'seeded_union_complete': bool(np.array_equal(split > 0, seeded_territory)),
        })
    proxy_start_id = next_cell_id
    for nucleus_id in unmatched_nucleus_ids.astype(int):
        nucleus_to_final_cell[nucleus_id] = next_cell_id; next_cell_id += 1
    proxy_stop_id = next_cell_id
    assert np.all(nucleus_to_final_cell[raw_nucleus_ids] > 0)
    assert all(record['seeded_union_complete'] for record in watershed_records)
    del cell_slices, nucleus_slices; gc.collect()

    for y0 in range(0, model_shape[0], ROW_CHUNK):
        y1 = min(model_shape[0], y0 + ROW_CHUNK)
        raw_nuclei = final_nuclei[y0:y1]
        mapped = nucleus_to_final_cell[raw_nuclei]
        final_cells[y0:y1][mapped > 0] = mapped[mapped > 0]
        raw_nuclei[:] = mapped
    final_cells.flush(); final_nuclei.flush()


## Validate before writing reconciled output


In [ ]:
if RUN_WATERSHED_RECONCILIATION:
    max_final_id = next_cell_id - 1
    final_cell_areas = np.zeros(max_final_id + 1, dtype=np.int64)
    final_nucleus_areas = np.zeros(max_final_id + 1, dtype=np.int64)
    mismatched_nuclear_pixels = 0
    for y0 in range(0, model_shape[0], ROW_CHUNK):
        y1 = min(model_shape[0], y0 + ROW_CHUNK)
        c = np.asarray(final_cells[y0:y1]); n = np.asarray(final_nuclei[y0:y1])
        mismatched_nuclear_pixels += int(np.count_nonzero((n > 0) & (c != n)))
        final_cell_areas += np.bincount(c.ravel(), minlength=max_final_id + 1)
        final_nucleus_areas += np.bincount(n.ravel(), minlength=max_final_id + 1)
    proxy_ids = np.arange(proxy_start_id, proxy_stop_id, dtype=np.int64)
    proxy_exact = np.array_equal(final_cell_areas[proxy_ids], final_nucleus_areas[proxy_ids])
    assert mismatched_nuclear_pixels == 0 and proxy_exact
    validation = {
        'all_raw_nuclei_assigned': bool(np.all(nucleus_to_final_cell[raw_nucleus_ids] > 0)),
        'mismatched_nuclear_pixels': mismatched_nuclear_pixels,
        'proxy_cells': int(len(proxy_ids)), 'all_proxy_cells_exact': bool(proxy_exact),
        'final_cells': int(np.count_nonzero(final_cell_areas[1:])),
        'final_nuclei': int(np.count_nonzero(final_nucleus_areas[1:])),
        'unseeded_parent_pixels_preserved': int(sum(r['unseeded_parent_pixels_preserved'] for r in watershed_records)),
    }
    display(validation)

    # Compact seam diagnostic: report foreground density in narrow bands around internal WSI windows.
    from instanseg.utils.tiling import _chops
    settings = raw.attrs['wsi_settings']; pad2 = settings['overlap'] + settings['detection_size']
    chops = _chops(model_shape, (settings['tile_size'], settings['tile_size']), overlap=2 * pad2)
    seam_rows = chops[0][1:]; seam_columns = chops[1][1:]
    seam_report = {
        'row_seam_cell_foreground_fraction': [float(np.mean(final_cells[max(0,y-2):min(model_shape[0],y+2)] > 0)) for y in seam_rows],
        'column_seam_cell_foreground_fraction': [float(np.mean(final_cells[:, max(0,x-2):min(model_shape[1],x+2)] > 0)) for x in seam_columns],
    }
    display(seam_report)


In [ ]:
if WRITE_RECONCILED_ZARR:
    if not RUN_WATERSHED_RECONCILIATION:
        raise RuntimeError('Run and validate reconciliation in this kernel first.')
    reconciled = zarr.open(
        str(RECONCILED_ZARR), mode='w', shape=(2, *model_shape),
        chunks=raw.chunks, dtype=np.uint32,
    )
    reconciled.attrs.update({
        'status': 'in_progress', 'source_unresolved_zarr': str(UNRESOLVED_ZARR),
        'source_instanseg_head': INSTANSEG_HEAD, 'planes': ['nuclei', 'cells'],
        'association_summary': association_summary, 'validation': validation,
    })
    for y0 in range(0, model_shape[0], ROW_CHUNK):
        y1 = min(model_shape[0], y0 + ROW_CHUNK)
        reconciled[0, y0:y1] = final_nuclei[y0:y1]
        reconciled[1, y0:y1] = final_cells[y0:y1]
    reconciled.attrs['status'] = 'complete'
    print('wrote', RECONCILED_ZARR)


## Optional streamed native-resolution export

Each output TIFF tile is nearest-neighbor sampled directly from the reconciled model grid. No dense native-resolution label array is allocated.


In [ ]:
def nearest_neighbor_tile_iterator(source, target_shape, tile_shape=(512, 512)):
    source_height, source_width = source.shape
    target_height, target_width = target_shape
    tile_height, tile_width = tile_shape
    for y0 in range(0, target_height, tile_height):
        y1 = min(target_height, y0 + tile_height)
        ys = np.minimum(np.arange(y0, y1) * source_height // target_height, source_height - 1)
        for x0 in range(0, target_width, tile_width):
            x1 = min(target_width, x0 + tile_width)
            xs = np.minimum(np.arange(x0, x1) * source_width // target_width, source_width - 1)
            tile = np.zeros(tile_shape, dtype=np.uint32)
            sampled = source.oindex[ys, xs] if hasattr(source, 'oindex') else source[np.ix_(ys, xs)]
            tile[:y1-y0, :x1-x0] = np.asarray(sampled, dtype=np.uint32)
            yield tile

def write_native_tiled_mask(path, source, target_shape):
    tifffile.imwrite(
        path, data=nearest_neighbor_tile_iterator(source, target_shape),
        shape=target_shape, dtype=np.uint32, tile=(512, 512),
        compression='zlib', bigtiff=True, photometric='minisblack',
    )

if WRITE_NATIVE_TIFFS:
    reconciled = zarr.open(str(RECONCILED_ZARR), mode='r')
    assert reconciled.attrs['status'] == 'complete'
    write_native_tiled_mask(FINAL_NUCLEAR_TIFF, reconciled[0], native_shape)
    write_native_tiled_mask(FINAL_CELL_TIFF, reconciled[1], native_shape)
    print({'nuclear': str(FINAL_NUCLEAR_TIFF), 'cell': str(FINAL_CELL_TIFF)})


## Adoption gate

Do not replace canonical pipeline masks from this notebook. After representative-slide seam review and structural validation, move the resolver into the InstanSeg fork as a tested Zarr-to-Zarr function and only then consider pipeline integration.
